# 03. NLR workload feature analysis

This notebook selects evidence-based candidates for gray-box calibration and future DFL information estimation. It analyses scale-normalised dynamics, workload-condition sensitivity, and repeat-run uncertainty. It does **not** train a DFL model yet.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').is_dir() else cwd.parent
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from nlr_data.feature_analysis import run_feature_analysis

INPUT_PATH = PROJECT_ROOT / 'outputs' / 'signal_validation' / 'workload_features.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'feature_analysis'
result = run_feature_analysis(INPUT_PATH, OUTPUT_DIR)
features = result['features']
coverage = result['condition_coverage']
sensitivity = result['sensitivity']
condition_information = result['condition_information']
repeatability = result['repeatability']
print(f'Profiles: {len(features):,}; conditions: {len(condition_information):,}')

## 1. Dataset and condition-variable coverage

Condition variables differ by workload family. Sensitivity is therefore evaluated within each family instead of pooling incompatible experiments.

In [ ]:
profile_overview = (
    features.groupby('workload_type')
    .agg(
        profiles=('profile_id', 'size'),
        conditions=('condition_id', 'nunique'),
        median_duration_s=('duration_s', 'median'),
        measurement_scope=('measurement_scope', lambda s: ', '.join(sorted(s.dropna().astype(str).unique()))),
        measurement_origin=('measurement_origin', lambda s: ', '.join(sorted(s.dropna().astype(str).unique()))),
    )
    .reset_index()
)
display(profile_overview)
display(coverage.loc[coverage['non_null_count'].gt(0)].reset_index(drop=True))

## 2. Scale-normalised workload dynamics

Per-unit-per-second ramps divide dP/dt by each profile's mean power. This permits a first comparison across server-, cluster-, and facility-scale traces, while preserving absolute power separately for physical scaling.

In [ ]:
normalised_columns = [
    'coefficient_of_variation', 'peak_to_average_ratio',
    'p95_abs_ramp_pu_per_s', 'max_ramp_up_pu_per_s',
    'max_ramp_down_pu_per_s', 'burst_fraction'
]
summary = (
    features.groupby('workload_type')[normalised_columns]
    .agg(['median', lambda s: s.quantile(0.05), lambda s: s.quantile(0.95)])
)
summary.columns = [f'{feature}_{stat}' for feature, stat in summary.columns]
display(summary.reset_index())

In [ ]:
plot_data = [
    features.loc[features['workload_type'].eq(name), 'p95_abs_ramp_pu_per_s'].dropna().to_numpy()
    for name in sorted(features['workload_type'].unique())
]
labels = sorted(features['workload_type'].unique())
fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(plot_data, labels=labels, showfliers=False)
ax.set_yscale('log')
ax.set_ylabel('P95 |dP/dt| / mean power [pu/s] (log scale)')
ax.set_title('Normalised ramp distributions by workload family')
ax.tick_params(axis='x', rotation=25)
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Workload-condition sensitivity

Spearman correlation screens condition variables that may be useful gray-box inputs. These values are exploratory, not causal: some metadata variables are coupled by experiment design, and whole-facility groups have only four points.

In [ ]:
top_sensitivity = (
    sensitivity.sort_values(
        ['workload_type', 'absolute_spearman_rho'], ascending=[True, False]
    )
    .groupby('workload_type', group_keys=False)
    .head(8)
)
display(top_sensitivity[[
    'workload_type', 'condition_variable', 'feature',
    'observation_count', 'unique_input_count', 'spearman_rho', 'p_value'
]])

In [ ]:
examples = [
    ('training_or_finetuning', 'nodes', 'mean_power_w'),
    ('offline_inference', 'batch_size', 'max_ramp_down_pu_per_s'),
    ('online_inference_finite', 'request_rate', 'p95_abs_ramp_pu_per_s'),
    ('online_inference_rate', 'request_rate', 'p95_abs_ramp_pu_per_s'),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (workload, x_column, y_column) in zip(axes.flat, examples):
    subset = features.loc[features['workload_type'].eq(workload), [x_column, y_column]].dropna()
    ax.scatter(subset[x_column], subset[y_column], s=14, alpha=0.35)
    grouped = subset.groupby(x_column)[y_column].median()
    ax.plot(grouped.index, grouped.values, color='black', linewidth=1.5, label='condition median')
    ax.set_title(workload)
    ax.set_xlabel(x_column)
    ax.set_ylabel(y_column)
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Repeat-run uncertainty

Within-condition CV measures repeatability under nominally identical metadata. A low mean-power CV with a higher ramp/burst CV supports estimating both a point level and an uncertainty/ramp descriptor rather than treating the trace as deterministic.

In [ ]:
focus = repeatability.loc[repeatability['feature'].isin([
    'mean_power_w', 'coefficient_of_variation',
    'p95_abs_ramp_pu_per_s', 'max_ramp_up_pu_per_s',
    'max_ramp_down_pu_per_s', 'peak_to_average_ratio'
])].copy()
display(focus.sort_values(['workload_type', 'feature']))

pivot = focus.pivot(index='workload_type', columns='feature', values='median_within_condition_cv')
display(pivot.style.format('{:.3f}').background_gradient(cmap='YlOrRd', axis=None))

## 5. Candidate targets and future DFL information

The following registries make the next modelling decision explicit. Primary candidates represent level, normal high ramps, extreme directional ramps, and burst occupancy. Temporal features are secondary until their stability and physical interpretation are checked.

In [ ]:
graybox_candidates = pd.read_csv(OUTPUT_DIR / 'graybox_target_candidates.csv')
dfl_candidates = pd.read_csv(OUTPUT_DIR / 'dfl_information_candidates.csv')
display(graybox_candidates)
display(dfl_candidates)

## Interpretation for the next stage

1. Use workload-specific condition inputs rather than one universal metadata vector.
2. Start the gray-box workload block with mean power, coefficient of variation, P95 normalised ramp, directional maximum ramps, and burst fraction.
3. Use repeated conditions to learn or validate P05–P95 uncertainty intervals.
4. Keep `measurement_scope` and `measurement_origin` as mandatory provenance flags. Most NLR traces describe IT/server or cluster power; the four whole-facility traces are simulated.
5. Do not send NLR values directly as PCC power. First propagate them through the facility gray-box mapping for cooling, UPS, auxiliary loads, ESS, and TES.
6. The next coding milestone is a workload-conditioned baseline estimator and a train/validation/test split that keeps repeated runs and conditions leakage-safe.